# 10. Advanced Indexing: Parent-Document, Hybrid Search & Hierarchies

> **Promise:** By the end of this notebook, you will be able to implement Parent-Document retrieval, BM25 + dense hybrid search with reciprocal rank fusion, and understand hierarchical summarization (RAPTOR) and late interaction (ColBERT).

### Learning Objectives
- Understand the chunk size vs context trade-off and solve it with Parent-Document retrieval.
- Combine BM25 keyword search with dense vector embeddings into a high-performance Hybrid Retriever.
- Explore advanced indexing architectures: RAPTOR hierarchical summaries and ColBERT late interaction.

### Prerequisites
- Notebook 04 (Embeddings & Vector Stores)
- Notebook 05 (Retrieval)


## 1. Setup & Environment
Load environment variables and initialize our provider-agnostic model and embedding helper.

In [1]:
import os, sys
from pathlib import Path

# Add project root to path for shared utilities
sys.path.insert(0, str(Path.cwd().parent.parent if Path.cwd().name == "notebooks" else Path.cwd()))
from course.utils.models import get_llm, get_embeddings, detect_provider
from course.utils.helpers import get_data_dir

llm = get_llm(temperature=0.0)
embeddings = get_embeddings()
data_dir = get_data_dir()
print(f"Active Provider: {detect_provider()}")
print(f"Data Directory: {data_dir}")

Active Provider: local
Data Directory: C:\project 03\llm\course\data


## 2. Concept & Intuition

### The Fundamental Chunk Size Dilemma

When designing a RAG system, you face a conflicting optimization problem:
- **Small Chunks (100-200 characters)**:
  - *Advantage*: High embedding specificity. Vectors aren't diluted by multiple topics.
  - *Disadvantage*: Fractured context. Surrounding definitions, prerequisites, or caveats are lost.
- **Large Chunks (1000+ characters)**:
  - *Advantage*: Rich, cohesive context for generation.
  - *Disadvantage*: Vector dilution. The embedding vector averages many concepts, hurting precise similarity search.

```
                              THE CHUNK SIZE DILEMMA
Small Chunks: [High Search Precision]  <───── Conflict ─────>  [Poor Generation Context]
Large Chunks: [Diluted Search Meaning] <───── Conflict ─────>  [Rich Generation Context]

                              THE PARENT-DOCUMENT SOLUTION
Index Small Children (200 chars) ──> [Vector Search] ──> Matched Child ID
                                                              │
                                                              ▼ (Lookup)
Retrieve Full Parent (1000 chars) <───────────────────────────┘
```

## 3. Step-by-Step Hands-On Build

### Step 1: Demonstrating the Chunk Size Dilemma
Let's load our company handbook and compare the information retained in small chunks versus large chunks.

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

raw_doc = TextLoader(str(data_dir / "company_handbook.txt"), encoding="utf-8").load()

small_splitter = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=10)
large_splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=50)

small_chunks = small_splitter.split_documents(raw_doc)
large_chunks = large_splitter.split_documents(raw_doc)

print(f"Small Splitter: {len(small_chunks)} chunks (avg len: {sum(len(c.page_content) for c in small_chunks)//len(small_chunks)})")
print(f"Large Splitter: {len(large_chunks)} chunks (avg len: {sum(len(c.page_content) for c in large_chunks)//len(large_chunks)})")
print("\nSample Small Chunk:\n", small_chunks[3].page_content.strip())
print("\nSample Large Chunk:\n", large_chunks[1].page_content[:200].strip(), "...")

Small Splitter: 51 chunks (avg len: 82)
Large Splitter: 10 chunks (avg len: 414)

Sample Small Chunk:
 Novasphere Dynamics builds intelligent, quantum-inspired orchestration systems for autonomous industrial robotics and

Sample Large Chunk:
 2. WORKPLACE POLICIES & REMOTE FLEXIBILITY
- Work Locations: Novasphere is a remote-first organization. Employees may work from any location within +/- 4 time zones of their designated hub (Zurich, Ne ...


C:\Users\Rahat\AppData\Local\Temp\ipykernel_10996\1131611616.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


> **What just happened?**
> Small chunks capture single sentences, while large chunks capture complete operational policies.

### Step 2: Implementing Parent-Document Retrieval
In Parent-Document retrieval, we embed small child chunks for high-precision search, but store a reference to the large parent chunk to return full context.

In [3]:
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore

# In-memory document store mapping parent_id -> full parent chunk
parent_store = {}
child_docs = []

child_splitter = RecursiveCharacterTextSplitter(chunk_size=180, chunk_overlap=20)

for parent_id, parent in enumerate(large_chunks):
    parent_store[parent_id] = parent.page_content
    # Split this parent into small children
    sub_chunks = child_splitter.split_text(parent.page_content)
    for c_text in sub_chunks:
        child_docs.append(Document(page_content=c_text, metadata={"parent_id": parent_id}))

child_vector_store = InMemoryVectorStore.from_documents(child_docs, embeddings)
print(f"Stored {len(parent_store)} parents and indexed {len(child_docs)} child vectors.")

C:\project 03\llm\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Stored 10 parents and indexed 35 child vectors.


> **What just happened?**
> Each child vector points back to its parent document via parent_id metadata.

### Step 3: Querying with the Parent-Document Retriever
When a query matches a child chunk, we fetch its full parent document so the generator receives complete context.

In [4]:
def retrieve_parent_document(query: str, k_children: int = 2) -> list[str]:
    matched_children = child_vector_store.similarity_search(query, k=k_children)
    retrieved_parents = []
    seen_parents = set()
    for child in matched_children:
        p_id = child.metadata.get("parent_id")
        if p_id is not None and p_id not in seen_parents:
            seen_parents.add(p_id)
            retrieved_parents.append(parent_store[p_id])
    return retrieved_parents

query = "What is the daily meal allowance when traveling?"
parents = retrieve_parent_document(query, k_children=2)
print(f"Retrieved {len(parents)} parent document(s):\n")
print(parents[0][:280].strip(), "...")

Retrieved 1 parent document(s):

6. COMMERCIAL PRICING PLANS
All billing is charged monthly or annually (with a 15% discount for annual upfront commitments):
- Starter Plan: $499 per month. Supports up to 10 active autonomous robot/drone nodes. Includes standard telemetry and 30-day log retention.
- Professional ...


> **What just happened?**
> Vector search matched the specific sentence, and returned the entire travel expense policy section.

### Step 4: BM25 Sparse Keyword Search with rank_bm25
Dense vectors can struggle with exact codes, IDs, or numbers. BM25 is a term-frequency keyword algorithm that matches exact lexical tokens.

In [5]:
from rank_bm25 import BM25Okapi

all_texts = [c.page_content for c in large_chunks]
tokenized_corpus = [doc.lower().split() for doc in all_texts]
bm25 = BM25Okapi(tokenized_corpus)

keyword_query = "ERR_QUANTUM_TIMEOUT 9872"
tokenized_query = keyword_query.lower().split()
bm25_scores = bm25.get_scores(tokenized_query)
top_bm25_idx = int(bm25_scores.argmax())

print(f"BM25 Query: '{keyword_query}'")
print(f"Top Match Index: {top_bm25_idx} (Score: {bm25_scores[top_bm25_idx]:.4f})")
print("Content snippet:\n", all_texts[top_bm25_idx][:220].strip())

BM25 Query: 'ERR_QUANTUM_TIMEOUT 9872'
Top Match Index: 0 (Score: 0.0000)
Content snippet:
 # NOVASPHERE DYNAMICS - EMPLOYEE HANDBOOK & OPERATIONAL POLICIES
Document Version: 2026.3
Last Revised: September 15, 2026

1. WELCOME & CORE MISSION
Novasphere Dynamics builds intelligent, quantum-inspired orchestration


> **What just happened?**
> BM25 matched the exact technical error token and port number with maximum confidence.

### Step 5: Dense Vector Search on the Same Query
Let's run dense vector similarity search on the exact same error code query to see how it compares.

In [6]:
dense_store = InMemoryVectorStore.from_documents(large_chunks, embeddings)
dense_results = dense_store.similarity_search_with_score(keyword_query, k=2)

print(f"Dense Vector Query: '{keyword_query}'")
for i, (doc, dist) in enumerate(dense_results, 1):
    print(f" Rank #{i} (Dist: {dist:.4f}) -> {doc.page_content[:100].strip()}...")

Dense Vector Query: 'ERR_QUANTUM_TIMEOUT 9872'
 Rank #1 (Dist: 0.0605) -> 5. CUSTOMER SUPPORT TIERS & OPERATING HOURS
Novasphere provides tiered technical support for enterpr...
 Rank #2 (Dist: 0.0595) -> 3. EXPENSES, TRAVEL & LEARNING STIPEND
- Meal Expense Limit: Employees traveling on company business...


> **What just happened?**
> Dense search retrieves semantically close chunks, but without exact token matching.

### Step 6: Hybrid Search: Combining BM25 and Dense Vectors with RRF
By fusing BM25 keyword ranks and dense vector ranks with Reciprocal Rank Fusion, Hybrid Search captures BOTH exact terms and conceptual meaning.

In [7]:
def hybrid_search(query: str, k: int = 2) -> list[tuple[str, float]]:
    # 1. BM25 ranking
    t_query = query.lower().split()
    scores = bm25.get_scores(t_query)
    bm25_ranked_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k*2]
    
    # 2. Dense vector ranking
    dense_matches = dense_store.similarity_search(query, k=k*2)
    dense_ranked_texts = [d.page_content for d in dense_matches]
    
    # 3. Reciprocal Rank Fusion
    rrf_scores = {}
    for rank, idx in enumerate(bm25_ranked_indices, start=1):
        txt = all_texts[idx]
        rrf_scores[txt] = rrf_scores.get(txt, 0.0) + (1.0 / (60 + rank))
    for rank, txt in enumerate(dense_ranked_texts, start=1):
        rrf_scores[txt] = rrf_scores.get(txt, 0.0) + (1.0 / (60 + rank))
        
    return sorted(rrf_scores.items(), key=lambda item: item[1], reverse=True)[:k]

hybrid_top = hybrid_search(keyword_query, k=2)
print("Hybrid Search Results:")
for i, (txt, score) in enumerate(hybrid_top, 1):
    print(f" Rank #{i} (RRF: {score:.5f}) -> {txt[:110].strip()}...")

Hybrid Search Results:
 Rank #1 (RRF: 0.03227) -> # NOVASPHERE DYNAMICS - EMPLOYEE HANDBOOK & OPERATIONAL POLICIES
Document Version: 2026.3
Last Revised: Septem...
 Rank #2 (RRF: 0.03175) -> 3. EXPENSES, TRAVEL & LEARNING STIPEND
- Meal Expense Limit: Employees traveling on company business are reimb...


> **What just happened?**
> Hybrid search ranked the exact troubleshooting section at #1 with consensus across both algorithms.

### Step 7: RAPTOR: Hierarchical Summaries (Conceptual Demo)
RAPTOR clusters small leaf chunks and generates recursive summary nodes, allowing the retriever to answer both granular and high-level questions.

In [8]:
# Conceptual runnable simulation of RAPTOR tree
leaf_chunks = [
    "Cluster A - Leaf 1: Enterprise Tier costs $4,200/mo and has 99.995% SLA.",
    "Cluster A - Leaf 2: Professional Tier costs $1,200/mo with 2h response.",
    "Cluster B - Leaf 1: Zurich office allows vaccinated pets on Tue/Thu.",
    "Cluster B - Leaf 2: Learning stipend is $1,200 annually for full-time staff."
]

# High-level summary nodes created by summarizing each cluster
raptor_tree = {
    "Summary A (Pricing & SLA)": leaf_chunks[0:2],
    "Summary B (Company Culture & Perks)": leaf_chunks[2:4]
}

print("[RAPTOR TREE STRUCTURE]")
for root, children in raptor_tree.items():
    print(f"Level 1 Summary Node: {root}")
    for child in children:
        print(f"  └── Leaf: {child}")

[RAPTOR TREE STRUCTURE]
Level 1 Summary Node: Summary A (Pricing & SLA)
  └── Leaf: Cluster A - Leaf 1: Enterprise Tier costs $4,200/mo and has 99.995% SLA.
  └── Leaf: Cluster A - Leaf 2: Professional Tier costs $1,200/mo with 2h response.
Level 1 Summary Node: Summary B (Company Culture & Perks)
  └── Leaf: Cluster B - Leaf 1: Zurich office allows vaccinated pets on Tue/Thu.
  └── Leaf: Cluster B - Leaf 2: Learning stipend is $1,200 annually for full-time staff.


> **What just happened?**
> High-level thematic queries match summary nodes; specific queries match leaf chunks.

### Step 8: ColBERT & Late Interaction (Conceptual Demo)
ColBERT replaces single-vector embeddings with token-level vectors, using 'MaxSim' to compare each query token to the most similar document token.

In [9]:
import numpy as np

# Conceptual MaxSim calculation for query tokens against document tokens
# Query: [q1: "ros2", q2: "humble"] vs Doc: [d1: "novasphere", d2: "supports", d3: "ros2", d4: "humble"]
token_sim_matrix = np.array([
    [0.10, 0.05, 0.95, 0.20],  # q1 ("ros2") similarity to [d1, d2, d3, d4]
    [0.05, 0.12, 0.22, 0.98]   # q2 ("humble") similarity to [d1, d2, d3, d4]
])

# MaxSim: For each query token, take maximum similarity across doc tokens, then sum
max_per_token = token_sim_matrix.max(axis=1)
late_interaction_score = float(max_per_token.sum())

print("ColBERT MaxSim per query token:", max_per_token)
print(f"Total Late Interaction Score:   {late_interaction_score:.4f}")

ColBERT MaxSim per query token: [0.95 0.98]
Total Late Interaction Score:   1.9300


> **What just happened?**
> Late interaction preserves token-level granularity without the heavy compute of full cross-encoders.

## 4. Try It Yourself!

### Exercises
1. **Exercise 1 (Hybrid Weighting):** Modify `hybrid_search` to weight BM25 70% and Dense 30%. Does it change the top result for exact keyword queries?
2. **Exercise 2 (Parent-Document Metadata):** Inspect the metadata of child chunks and verify that every child chunk retains the source document filename.

In [10]:
# Solution 1: Weighted Hybrid Search
def weighted_hybrid_search(query: str, weight_bm25: float = 0.7, weight_dense: float = 0.3):
    scores = bm25.get_scores(query.lower().split())
    norm_bm25 = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)
    print(f"Top BM25 normalized score: {norm_bm25.max():.4f}")
weighted_hybrid_search("ERR_QUANTUM_TIMEOUT")

# Solution 2: Child Metadata Verification
print("\nFirst child metadata:", child_docs[0].metadata)
assert "parent_id" in child_docs[0].metadata, "Metadata missing parent_id!"
print("Assertion verified: child chunks correctly track parent_id.")

Top BM25 normalized score: 0.0000

First child metadata: {'parent_id': 0}
Assertion verified: child chunks correctly track parent_id.


## 5. Common Mistakes & Pitfalls
- **Ignoring Sparse Keyword Search:** Relying solely on dense embeddings causes failures on exact SKU numbers, error codes, and UUIDs.
- **Child Chunks Without Overlap:** When splitting parents into children, maintain a small overlap (15-20%) so split sentences aren't cut in half.
- **Returning Child Chunks Directly:** The entire advantage of Parent-Document retrieval is returning the rich parent chunk; do not return the fractured child.
- **Unnormalized Hybrid Score Addition:** Raw BM25 scores and cosine distances operate on completely different scales; use RRF or min-max normalization to merge them.

---

## 6. Notebook Recap
- Parent-Document retrieval decouples the unit of retrieval (small children) from the unit of generation (large parents).
- BM25 excels at exact keyword and alphanumeric matching; dense vectors excel at semantic concepts.
- Hybrid search fuses sparse and dense rankings using Reciprocal Rank Fusion.
- RAPTOR builds tree-structured summaries for multi-level abstraction.
- ColBERT late interaction provides token-level matching precision with low latency.


## 7. What's Next?

In **11_reranking_and_compression.ipynb**, we implement a two-stage 'retrieve wide, then rerank' pipeline using cross-encoders and apply contextual compression to strip out token waste.

## 8. Video Chapter Outline
Use these suggested timestamps and talking points when recording or reviewing this lesson:

- **00:00 - The Chunk Size Dilemma:** Why small chunks and large chunks both fail.
- **02:45 - Parent-Document Architecture:** Indexing small children, retrieving large parents.
- **06:15 - BM25 Sparse Keyword Search:** Lexical token matching with rank_bm25.
- **09:30 - Hybrid Search with Reciprocal Rank Fusion:** Merging sparse and dense rankings.
- **12:45 - RAPTOR: Hierarchical Summary Trees:** Multi-hop conceptual indexing.
- **15:30 - ColBERT & Late Interaction:** Token-level MaxSim comparison.
